# RobustWorld: Wan 2.1 VACE on Colab

Predicts the second half (target) of each eval clip from its first half (context), then builds a
comparison grid against the ground truth. The sample is 5 clips committed in `data/eval/sample5`.

**Runtime:** GPU. The 1.3B model fits a free **T4** (fp16); an **L4** or **A100** runs bf16 and is faster. Expect a few minutes per clip on a T4. Switch to `--model-id Wan-AI/Wan2.1-VACE-14B-diffusers` for the 14B model (A100 80 GB).

**Secrets** (key icon in the left sidebar, then enable notebook access), if any:
- `GITHUB_TOKEN`: not needed while the repo is public (only if it is made private again).


Predictions are saved to Google Drive under `MyDrive/RobustWorld/predictions/`. The other model's
notebook writes to the same place, so the grid at the end includes every model that has been run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = "/content/RobustWorld"
if not os.path.exists(REPO):
    try:  # only needed if the repo is private again
        url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/FinPThomas/RobustWorld.git"
    except Exception:
        url = "https://github.com/FinPThomas/RobustWorld.git"
    subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://github.com/FinPThomas/RobustWorld.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip install -q -e . -r models/wan21/requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PRED_ROOT = "/content/drive/MyDrive/RobustWorld/predictions"
GRID_ROOT = "/content/drive/MyDrive/RobustWorld/grids"
SAMPLE = "data/eval/sample5"
os.makedirs(PRED_ROOT, exist_ok=True)

## Run Wan 2.1 VACE

Gets all 16 context frames and generates the next 17 (Wan needs 4n+1 frames); the first 16 are
compared with the ground truth. On a T4 add `--offload` if it runs out of memory.

In [ ]:
!python models/wan21/predict.py --sample {SAMPLE} --out {PRED_ROOT}/sample5/wan21 --steps 50

## Comparison grid

Columns: ground truth, then every model with predictions in Drive. Rows: the input each source
was given (frames a model didn't see are dimmed), then its output. The context plays first,
then the predictions.

In [ ]:
import pathlib
order = ["ltx_video", "cosmos_predict2", "wan21"]            # column order; add new models here
have = [m for m in order if (pathlib.Path(PRED_ROOT) / "sample5" / m).exists()]
out = f"{GRID_ROOT}/sample5/{'+'.join(have)}"
!robustworld-eval grid --sample {SAMPLE} --pred-root {PRED_ROOT} --models {" ".join(have)} --out {out}

In [ ]:
from IPython.display import Video, display
display(Video(f"{out}/all.mp4", embed=True, width=900))